In [1]:
# =============================================================================
# [실습 목표] 세 가지 방법으로 감성 분류 성능 비교
#
#  1. Zero-shot Prompting  : 예시 없이 모델에게 바로 질문
#  2. Few-shot Prompting   : 몇 가지 레이블된 예시를 프롬프트에 포함
#  3. Full Fine-tuning     : 모델 전체 가중치를 학습 데이터로 업데이트
#
# [비교 기준]
#  - 성능 (Accuracy)
#  - 자원 소모 (학습 시간, 업데이트된 파라미터 비율)
#
# [사용 모델] google/flan-t5-small
#  - T5 계열의 seq2seq(인코더-디코더) 소형 모델 (~77M 파라미터)
#  - 텍스트 → 텍스트 형태로 분류 결과를 출력
# =============================================================================
 

In [2]:
# 1. 라이브러리 임포트
import time
import torch
import pandas as pd
from datasets import Dataset    # HuggingFace data
from transformers import (
    AutoTokenizer,              # 텍스트 -> 토큰 ID 변환기
    AutoModelForSeq2SeqLM,      # seq2seq(인코더-디코더) 모델
    Seq2SeqTrainer,             # seq2seq 전용 학습기
    Seq2SeqTrainingArguments,   # 학습 하이퍼파라미터 설정
    DataCollatorForSeq2Seq      # 배치, 패딩 자동 처리 콜레이터
)

# 2. 디바이스 설정
# GPU 있으면 cuda, 없으면 cpu
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'사용 디바이스: {device}')

c:\Users\Playdata\miniconda3\envs\llm\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


사용 디바이스: cpu


In [3]:
# 3. 데이터셋 정의
train_data = [   # 학습 데이터 15개 — Zero-shot/Few-shot에는 사용 안 하고, Fine-tuning 학습에만 사용
    {"text": "This movie was absolutely fantastic! The acting was superb.",          "label": "positive"},
    {"text": "I wasted two hours of my life. Terrible plot and cheap CGI.",          "label": "negative"},
    {"text": "A true masterpiece of cinema. Highly recommend it to everyone.",       "label": "positive"},
    {"text": "Extremely boring and predictable. I fell asleep halfway through.",     "label": "negative"},
    {"text": "The cinematography was beautiful, and the story was touching.",        "label": "positive"},
    {"text": "Horrible acting and bad direction. Do not watch this film.",           "label": "negative"},
    {"text": "Brilliant performance by the lead actor, a must-watch.",               "label": "positive"},
    {"text": "The plot made no sense, and the characters were annoying.",            "label": "negative"},
    {"text": "I loved the soundtrack and the emotional depth of this movie.",        "label": "positive"},
    {"text": "Total waste of money. I regret watching this garbage.",                "label": "negative"},
    {"text": "Very engaging storyline with great character development.",            "label": "positive"},
    {"text": "Slow, dull, and lacks any real substance.",                            "label": "negative"},
    {"text": "An amazing adventure that kept me on the edge of my seat.",            "label": "positive"},
    {"text": "A complete disappointment. The trailer was much better than the film.","label": "negative"},
    {"text": "Wonderfully written and beautifully executed. A delight to watch.",   "label": "positive"},
]

test_data = [   # 평가 데이터 5개 — 세 방법의 정확도 측정에 공통 사용
    {"text": "The movie was mediocre, but the ending was spectacular!",    "label": "positive"},
    {"text": "Worst movie I have seen this year. Avoid at all costs.",     "label": "negative"},
    {"text": "A refreshing and delightful comedy that had me laughing all night.", "label": "positive"},
    {"text": "The acting was flat and the script felt extremely forced.",  "label": "negative"},
    {"text": "An absolute gem of a film with outstanding visuals.",        "label": "positive"},
]

print(f"학습 데이터 크기: {len(train_data)}개")
print(f"평가 데이터 크기: {len(test_data)}개")

학습 데이터 크기: 15개
평가 데이터 크기: 5개


In [ ]:
# 4. 모델 & 토크나이저 로드
model_name = 'google/flan-t5-small'     # HuggingFace Hub model ID

# 토크나이저 : 텍스트를 모델이 이해하는 토큰 ID 시퀀스로 변환 (모델과 동일한 어휘 사전)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 모델 : flan-t5-small (인코도-디코더 seq2seq 구조, 약 77M 파라미터)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model.to(device)    # 모델 가중치를 GPU 또는 CPU메모리로 이동

In [5]:
# 5. 파라미터 수 확인 함수
# 전체 파라미터 / 학습 가능한 파라미터 / 비율(%) 반환
# -> Full Fine-tuning이 얼마나 많은 가중치를 업데이트하는지 확인하는 용도
def get_trainable_params(model):
    all_param = 0
    trainable_params = 0
    for _, param in model.named_parameters():   # 모든 레이어의 파라미터 순회
        all_param += param.numel()              # 전체 파라미터 수 누적
        if param.requires_grad:                 # gradient 계산 대상(학습 가능)인 파라미터만 카운트
            trainable_params += param.numel()
    return all_param, trainable_params, trainable_params / all_param*100

all_p, train_p, pct = get_trainable_params(model)
print(f"\n전체 파라미터: {all_p:,}")
print(f"학습 가능 파라미터: {train_p:,} ({pct:.2f}%)")


전체 파라미터: 76,961,152
학습 가능 파라미터: 76,961,152 (100.00%)


In [6]:
# 6. 추론 함수 정의
# prompt를 입력받아 모델이 생성한 텍스트(예측 레이블)를 반환
def generate_prediction(prompt, model, tokenizer, max_new_tokens=10):
    inputs = tokenizer(prompt, return_tensors='pt').to(device)  # 텍스트 -> 토큰 텐서, 디바이스 이동
    with torch.no_grad():       # 추론 시 gradient 계산 비활성화 (메모리 절약)
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)   # 최대 10개 토큰 생성
    pred_text = tokenizer.decode(outputs[0], skip_special_tokens=True).strip().lower() # 토큰 ID -> 문자열, 소문자 정리
    return pred_text

In [10]:
# 7. 평가 함수 정의
# test_data 전체를 순회하면 Accuracy와 결과 DataFrame을 반환
# prompt_type: 'zero_shot' | 'few_shot' | 'fine_tuned' 세 가지 모드 지원
def evaluate_model(model, tokenizer, test_data, prompt_type='zero_shot'):
    correct = 0
    results = []

    # few_shot 프롬프트에 포함할 레이블된 예시 3개
    # 모델에게 "입력 -> 출력 형식"을 시범으로 보여주는 역할
    few_shot_examples = (
        "Review: Brilliant performance by the lead actor, a must-watch. \nSentiment: positive\n\n"
        "Review: The plot made no sense, and the characters were annoying. \nSentiment: negative\n\n"
        "Review: I loved the soundtrack and the emotional depth of this movie. \nSentiment: positive\n\n"
    )

    for item in test_data:
        text = item['text']
        true_label = item['label']

        # 프롬프트 구성 (방식에 따라 다르게 구성)
        if prompt_type == 'zero_shot':
            # Zero-shot: 예시 없이 바로 질문 - 모델의 사전 학습 지식만으로 판단
            prompt = f'Review:{text} \nSentiment (positive or negative)'
        elif prompt_type == 'few_shot':
            # Few-shot: 레이블된 예시 3개를 앞에 붙여 문맥을 제공 — 별도 학습 없이 성능 향상 기대
            prompt = few_shot_examples + f'Review:{text} \nSentiment (positive or negative)'
        else:
            # Fine-tuned: 파인튜닝 후 평가 - Zero-shot과 동일한 형식이지만 가중치가 업데이트 된 상태
            prompt = f'Review:{text} \nSentiment (positive or negative)'

        pred_label = generate_prediction(prompt, model, tokenizer)

        # 모델 출력에서 positive/negative 키워드만 추출 (불필요한 토큰 제거)
        if 'positive' in pred_label:
            cleaned_pred = 'positive'
        elif 'negative' in pred_label:
            cleaned_pred = 'negative'
        else:
            cleaned_pred = pred_label   # 해당 없으면 원본 그대로 유지

        is_correct = (cleaned_pred == true_label)
        if is_correct:
            correct += 1

        results.append({
            'Text': text,
            'True Label': true_label,
            'Pred Label': pred_label,
            'Cleaned Pred': cleaned_pred,
            'Correct': is_correct
        })

    accuracy = correct / len(test_data)    # 정확도 = 정답 수 / 전체 샘플 수
    return accuracy, pd.DataFrame(results)

In [11]:
# 8. Zero-shot Prompting 평가
# 학습 없이, 사전 학습된 모델 그대로 사용
print("\n" + "="*60)
print("[ 1. Zero-shot Prompting ]")
print("="*60)
zero_shot_acc, zero_shot_df = evaluate_model(model, tokenizer, test_data, 'zero_shot')
print(f"Zero-shot Accuracy: {zero_shot_acc:.2%}")
print(zero_shot_df[['Text', 'True Label', 'Cleaned Pred', 'Correct']].to_string())
zero_shot_df


[ 1. Zero-shot Prompting ]
Zero-shot Accuracy: 20.00%
                                                                 Text True Label                                     Cleaned Pred  Correct
0             The movie was mediocre, but the ending was spectacular!   positive            the movie is a great movie. the story    False
1              Worst movie I have seen this year. Avoid at all costs.   negative  i have seen this movie for years and have never    False
2  A refreshing and delightful comedy that had me laughing all night.   positive               i loved this movie. it was a great    False
3           The acting was flat and the script felt extremely forced.   negative                                         negative     True
4                 An absolute gem of a film with outstanding visuals.   positive                      this is a great film. it is    False


,Text,True Label,Pred Label,Cleaned Pred,Correct
0,"The movie was mediocre, but the ending was spe...",positive,the movie is a great movie. the story,the movie is a great movie. the story,False
1,Worst movie I have seen this year. Avoid at al...,negative,i have seen this movie for years and have never,i have seen this movie for years and have never,False
2,A refreshing and delightful comedy that had me...,positive,i loved this movie. it was a great,i loved this movie. it was a great,False
3,The acting was flat and the script felt extrem...,negative,negative,negative,True
4,An absolute gem of a film with outstanding vis...,positive,this is a great film. it is,this is a great film. it is,False


In [ ]:
# 9. Few-shot Prompting 평가